# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
import os
from dotenv import load_dotenv

load_dotenv()

PRICE_DATA = os.getenv("PRICE_DATA")



In [ ]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True)




For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Write your code below.
import dask.dataframe as dd


dd_prices = dd.read_parquet(parquet_files)

dd_feat = dd_prices.groupby("Ticker", group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1=x["Close"].shift(1),
        Adj_Close_lag_1=x["Adj Close"].shift(1),
        Returns=(x["Close"] / x["Close"].shift(1)) - 1,
        hi_lo_range=x["High"] - x["Low"]
    )
)




+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [5]:
# Write your code below.
df_stock_features = dd_feat.compute()


# Adding in 10-day moving average of returns
df_stock_features["ten_day_avg_returns"] = df_stock_features.groupby("Ticker")["Returns"].transform(lambda x: x.rolling(10).mean())


In [6]:
df_stock_features

Price,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range,ten_day_avg_returns
Ticker,,,,,,,,,,,,,
DOV,2017-01-03 00:00:00+00:00,53.950451,61.728596,62.625202,60.920841,61.639744,1637998.0,2017,NaN,NaN,NaN,1.704361,NaN
DOV,2017-01-04 00:00:00+00:00,54.204609,62.019386,62.075928,61.445881,61.882069,1165948.0,2017,61.728596,53.950451,0.004711,0.630047,NaN
DOV,2017-01-05 00:00:00+00:00,54.028114,61.817448,62.431339,61.074314,61.793217,1155178.0,2017,62.019386,54.204609,-0.003256,1.357025,NaN
DOV,2017-01-06 00:00:00+00:00,54.868214,62.778675,63.780293,62.390953,62.463650,3123350.0,2017,61.817448,54.028114,0.015549,1.389339,NaN
DOV,2017-01-09 00:00:00+00:00,54.162251,61.970921,62.924072,61.744751,62.705978,1271055.0,2017,62.778675,54.868214,-0.012867,1.179321,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
CTLT,2021-12-27 00:00:00+00:00,126.360001,126.360001,127.300003,124.949997,125.900002,452316.0,2021,124.959999,124.959999,0.011204,2.350006,0.003572
CTLT,2021-12-28 00:00:00+00:00,125.360001,125.360001,127.000000,125.055000,125.930000,335608.0,2021,126.360001,126.360001,-0.007914,1.945000,0.000688
CTLT,2021-12-29 00:00:00+00:00,128.080002,128.080002,128.520004,124.309998,125.059998,715711.0,2021,125.360001,125.360001,0.021698,4.210007,0.004860


In [7]:
df_stock_features["ten_day_avg_returns"].shape

(3173427,)

In [8]:
df_stock_features["ten_day_avg_returns"].isna().sum()

392322

In [9]:
df_stock_features["ten_day_avg_returns"].notna().sum()

2781105

In [10]:
df_stock_features["ten_day_avg_returns"].describe()

count    2.781105e+06
mean     4.855002e-03
std      1.225927e-01
min     -1.807723e-01
25%     -2.720791e-03
50%      7.011021e-04
75%      4.011962e-03
max      2.749911e+01
Name: ten_day_avg_returns, dtype: float64

In [11]:
df_stock_features.describe()


Price,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range,ten_day_avg_returns
count,2.792542e+06,2.792542e+06,2.792542e+06,2.792542e+06,2.792542e+06,2.792542e+06,3.173427e+06,2.792094e+06,2.792094e+06,2.791398e+06,2.792542e+06,2.781105e+06
mean,7.121454e+01,8.066665e+01,8.158663e+01,7.970649e+01,8.065681e+01,8.049651e+06,2.012044e+03,8.066056e+01,7.120813e+01,4.841445e-03,1.880143e+00,4.855002e-03
std,1.834912e+02,1.862905e+02,1.884262e+02,1.841289e+02,1.862596e+02,4.712249e+07,7.233401e+00,1.862676e+02,1.834673e+02,3.872132e-01,5.028896e+00,1.225927e-01
min,3.052100e-02,3.052100e-02,3.052100e-02,2.697900e-02,3.020800e-02,0.000000e+00,2.000000e+03,3.052100e-02,3.052100e-02,-9.992444e-01,0.000000e+00,-1.807723e-01
25%,1.659500e+01,2.362000e+01,2.394000e+01,2.328500e+01,2.361500e+01,9.398000e+05,2.006000e+03,2.361958e+01,1.659472e+01,-9.114583e-03,4.747086e-01,-2.720791e-03
50%,3.367482e+01,4.427000e+01,4.478000e+01,4.374000e+01,4.426000e+01,2.144200e+06,2.012000e+03,4.427000e+01,3.367368e+01,5.011963e-04,9.000015e-01,7.011021e-04
75%,7.182652e+01,8.343000e+01,8.431000e+01,8.250000e+01,8.343000e+01,5.092900e+06,2.018000e+03,8.342000e+01,7.182000e+01,1.016517e-02,1.814232e+00,4.011962e-03
max,9.924400e+03,9.924400e+03,9.964770e+03,9.794000e+03,9.914170e+03,9.230856e+09,2.025000e+03,9.924400e+03,9.924400e+03,2.749302e+02,6.572100e+02,2.749911e+01


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?

No, it wasn't necessary.
+ Would it have been better to do it in Dask? Why?

In this specific case, given the dataset size,  I believe Dask would have been more efficient and faster.
(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.